<a href="https://colab.research.google.com/github/LiQuinChing/DL-fakeNewsClassification/blob/Vihara-BiLSTM/DLDataPreprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================
# Clean & standardize news files (no extras)
# ============================================
import os, re, hashlib
import pandas as pd
from glob import glob

# Mount Drive in Colab if you haven't yet:
# from google.colab import drive
# drive.mount('/content/drive')

# ---- set your paths here ----
INPUT_ROOT  = "/content/drive/MyDrive/news_raw"      # contains "fake_and_real" and "fake news"
OUTPUT_ROOT = "/content/drive/MyDrive/news_clean1"    # cleaned files, mirrored folders
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# ---- helpers ----
TITLE_CANDIDATES = ["title","headline","headlines","article_title","story_title","subject"]
TEXT_CANDIDATES  = ["text","content","article","body","full_text","news","message","raw_text","paragraphs","desc","description"]
LABEL_CANDIDATES = ["label","labels","class","target","category","verdict","fake","is_fake","is_real","truth","type","news_type","label_name"]

def normalize_cols(df):
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    for c in df.columns:
        if pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip()
    return df

def pick_first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def clean_whitespace(s):
    if not isinstance(s, str):
        s = "" if pd.isna(s) else str(s)
    return re.sub(r"\s+", " ", s).strip()

def normalize_title_placeholders(s):
    if pd.isna(s):
        return ""
    s = str(s).strip()
    # turn placeholders like 'nan', 'no title', 'null', '-' into empty string
    return "" if re.fullmatch(r"(nan|none|no\s*title|notitle|n/?a|null|-)", s, flags=re.I) else s

def map_label_series(s):
    s = s.astype(str).str.strip().str.lower()
    mapping = {
        "real":1, "true":1, "legit":1, "genuine":1, "1":1, "yes":1, "y":1,
        "fake":0, "false":0, "bogus":0, "hoax":0,  "0":0, "no":0,  "n":0
    }
    out = s.map(mapping)
    if out.isna().any():
        out_num = pd.to_numeric(s[out.isna()], errors="coerce")
        out.loc[out.isna()] = out_num
    out = pd.to_numeric(out, errors="coerce")
    return out

def infer_label_from_path(relpath):
    """
    Use path hints for label when no label column:
    - any folder/file name containing real/true/legit -> 1
    - any containing fake/false/hoax                 -> 0
    - if inside a folder named exactly 'fake news'   -> 0
    """
    parts = [p.lower() for p in re.split(r"[\\/]", relpath)]
    joined = " ".join(parts)
    if "fake news" in joined:
        return 0
    # filename has strongest weight
    fname = parts[-1]
    def has(term): return any((term in p) or re.search(fr"\b{term}\b", p) for p in parts)
    if re.search(r"\breal\b", fname) or has("real") or has("true") or has("legit"):
        return 1
    if re.search(r"\bfake\b", fname) or has("fake") or has("false") or has("hoax"):
        return 0
    return None

def read_table_safely(path):
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".csv":
            return pd.read_csv(path, sep=None, engine="python", on_bad_lines="skip")
        if ext == ".tsv":
            return pd.read_csv(path, sep="\t", engine="python", on_bad_lines="skip")
        if ext in [".xlsx", ".xls"]:
            sheets = pd.read_excel(path, sheet_name=None)
            if not sheets:
                return None
            # choose the sheet with most rows
            return max(sheets.values(), key=lambda d: len(d))
        # fallback
        return pd.read_csv(path, engine="python", on_bad_lines="skip")
    except Exception as e:
        print(f"   [read error] {os.path.basename(path)}: {e}")
        return None

def standardize_df(df_raw, relpath_for_label_hint):
    df = normalize_cols(df_raw)

    text_col  = pick_first_existing(df, TEXT_CANDIDATES)
    title_col = pick_first_existing(df, TITLE_CANDIDATES)
    label_col = pick_first_existing(df, LABEL_CANDIDATES)

    # one-column file → treat as text
    if len(df.columns) == 1 and text_col is None:
        text_col = df.columns[0]

    # if still no text but title exists, use title as text
    if text_col is None and title_col is not None:
        text_col, title_col = title_col, None

    out = pd.DataFrame()
    out["text"]  = df[text_col] if (text_col and text_col in df.columns) else ""
    out["title"] = df[title_col] if (title_col and title_col in df.columns) else ""

    # label
    if label_col and label_col in df.columns:
        lbl = map_label_series(df[label_col])
        out["label"] = lbl
    else:
        inferred = infer_label_from_path(relpath_for_label_hint)
        out["label"] = inferred if inferred is not None else pd.NA

    # clean fields
    out["text"]  = out["text"].map(clean_whitespace)
    out["title"] = out["title"].map(normalize_title_placeholders)

    # keep only rows that have a label 0/1
    out["label"] = pd.to_numeric(out["label"], errors="coerce")
    out = out[out["label"].isin([0,1])].copy()

    # optional: drop empty texts (keep short if you prefer)
    out = out[out["text"].str.len() >= 1].copy()

    # final order & only required columns
    out = out[["title","text","label"]]
    return out

# ---- collect files ----
patterns = ["**/*.csv", "**/*.tsv", "**/*.xlsx", "**/*.xls"]
all_paths = []
for pat in patterns:
    all_paths.extend(glob(os.path.join(INPUT_ROOT, pat), recursive=True))
all_paths = sorted(all_paths)
print(f"Found {len(all_paths)} files under {INPUT_ROOT}")

# ---- process & save mirrored ----
count_written, count_skipped = 0, 0

for path in all_paths:
    rel = os.path.relpath(path, INPUT_ROOT)
    df_raw = read_table_safely(path)
    if df_raw is None or len(df_raw) == 0:
        print(f"– skip (empty/unreadable): {rel}")
        count_skipped += 1
        continue

    std = standardize_df(df_raw, rel)
    if std.empty:
        print(f"– skip (no usable rows after cleaning): {rel}")
        count_skipped += 1
        continue

    # mirror folder structure, write CSV with same base name
    out_path = os.path.join(OUTPUT_ROOT, os.path.splitext(rel)[0] + ".csv")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    std.to_csv(out_path, index=False)
    print(f"✓ wrote {len(std):5d} rows -> {out_path}")
    count_written += 1

print(f"\nDone. Files written: {count_written}, skipped: {count_skipped}")
print(f"Cleaned files live under: {OUTPUT_ROOT}")


Found 11 files under /content/drive/MyDrive/news_raw
✓ wrote 71349 rows -> /content/drive/MyDrive/news_clean1/fake/archive (1)/WELFake_Dataset.csv
✓ wrote 23196 rows -> /content/drive/MyDrive/news_clean1/fake/archive (2)/FakeNewsNet.csv
✓ wrote  2050 rows -> /content/drive/MyDrive/news_clean1/fake/archive (3)/news_articles.csv
✓ wrote    19 rows -> /content/drive/MyDrive/news_clean1/fake/archive (4)/fake.csv
✓ wrote 20000 rows -> /content/drive/MyDrive/news_clean1/fake/archive/fake_news_dataset.csv
– skip (no usable rows after cleaning): fake_and_real/archive (3)/fake.csv
– skip (no usable rows after cleaning): fake_and_real/archive (3)/real.csv
✓ wrote 22851 rows -> /content/drive/MyDrive/news_clean1/fake_and_real/archive (4)/Fake.csv
✓ wrote 21416 rows -> /content/drive/MyDrive/news_clean1/fake_and_real/archive (4)/True.csv
✓ wrote 22851 rows -> /content/drive/MyDrive/news_clean1/fake_and_real/archive/Fake.csv
✓ wrote 21416 rows -> /content/drive/MyDrive/news_clean1/fake_and_real/arc

In [ ]:
import os, re, pandas as pd

INPUT_DIR  = "/content/drive/MyDrive/news_raw/fake_and_real/archive (3)"
OUTPUT_DIR = "/content/drive/MyDrive/news_clean1/fake_and_real/archive (3)"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TITLE_CANDIDATES = ["title","headline","headlines","article_title","story_title","subject"]
TEXT_CANDIDATES  = ["text","content","article","body","full_text","news","message","raw_text","paragraphs","desc","description"]
LABEL_CANDIDATES = ["label","labels","class","target","category","verdict","fake","is_fake","is_real","truth","type","news_type","label_name"]

def normalize_cols(df):
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    for c in df.columns:
        if pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip()
    return df

def pick_first_existing(df, cands):
    for c in cands:
        if c in df.columns:
            return c
    return None

def clean_ws(s):
    if not isinstance(s, str):
        s = "" if pd.isna(s) else str(s)
    return re.sub(r"\s+", " ", s).strip()

def norm_title(s):
    if pd.isna(s): return ""
    s = str(s).strip()
    return "" if re.fullmatch(r"(nan|none|no\s*title|notitle|n/?a|null|-)", s, flags=re.I) else s

def map_label_series(s):
    s = s.astype(str).str.strip().str.lower()
    mapping = {"real":1,"true":1,"legit":1,"genuine":1,"1":1,"yes":1,"y":1,
               "fake":0,"false":0,"bogus":0,"hoax":0,"0":0,"no":0,"n":0}
    out = s.map(mapping)
    if out.isna().any():
        out_num = pd.to_numeric(s[out.isna()], errors="coerce")
        out.loc[out.isna()] = out_num
    return pd.to_numeric(out, errors="coerce")

def infer_label_from_filename(name):
    n = name.lower()
    if "real" in n or "true" in n or "legit" in n: return 1
    if "fake" in n or "false" in n or "hoax" in n: return 0
    return None

def read_table(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        return pd.read_csv(path, sep=None, engine="python", on_bad_lines="skip")
    if ext == ".tsv":
        return pd.read_csv(path, sep="\t", engine="python", on_bad_lines="skip")
    if ext in [".xlsx",".xls"]:
        sheets = pd.read_excel(path, sheet_name=None)
        if not sheets: return None
        return max(sheets.values(), key=lambda d: len(d))
    return pd.read_csv(path, engine="python", on_bad_lines="skip")

def standardize(df_raw, source_name):
    df = normalize_cols(df_raw)
    text_col  = pick_first_existing(df, TEXT_CANDIDATES)
    title_col = pick_first_existing(df, TITLE_CANDIDATES)
    label_col = pick_first_existing(df, LABEL_CANDIDATES)

    if len(df.columns) == 1 and text_col is None:
        text_col = df.columns[0]
    if text_col is None and title_col is not None:
        text_col, title_col = title_col, None

    out = pd.DataFrame()
    out["text"]  = df[text_col] if (text_col and text_col in df.columns) else ""
    out["title"] = df[title_col] if (title_col and title_col in df.columns) else ""

    if label_col and label_col in df.columns:
        out["label"] = map_label_series(df[label_col])
    else:
        inferred = infer_label_from_filename(source_name)
        out["label"] = inferred if inferred is not None else pd.NA

    out["text"]  = out["text"].map(clean_ws)
    out["title"] = out["title"].map(norm_title)
    out["label"] = pd.to_numeric(out["label"], errors="coerce")
    out = out[out["label"].isin([0,1])]
    out = out[out["text"].str.len() >= 1]
    return out[["title","text","label"]]

# List files we see (case-insensitive .csv/.xlsx/.xls)
seen = []
for name in os.listdir(INPUT_DIR):
    if os.path.isfile(os.path.join(INPUT_DIR, name)) and os.path.splitext(name)[1].lower() in (".csv",".xlsx",".xls",".tsv"):
        seen.append(name)
print("Files in archive (3):", seen)

# Process each
for name in seen:
    src = os.path.join(INPUT_DIR, name)
    df_raw = read_table(src)
    if df_raw is None or len(df_raw)==0:
        print("skip empty:", name); continue
    std = standardize(df_raw, name)
    if std.empty:
        print("skip no usable rows:", name); continue
    out = os.path.join(OUTPUT_DIR, os.path.splitext(name)[0] + ".csv")
    std.to_csv(out, index=False)
    print(f"✓ wrote {len(std)} rows -> {out}")


Files in archive (3): ['fake.csv']
skip no usable rows: fake.csv


In [ ]:
# ===============================
# Clean ONLY two folders in Colab
#   - news_raw/fake/archive (2)   (trust existing label)
#   - news_raw/fake/archive (4)   (use label if present, else default 0)
# ===============================
import os, re
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

INPUT_ROOT  = "/content/drive/MyDrive/news_raw"
OUTPUT_ROOT = "/content/drive/MyDrive/news_clean1"   # mirrored output; change to INPUT_ROOT to overwrite in-place

FOLDERS = [
    ("fake/archive (2)", True,  None),  # (subpath, trust_existing_label, default_label_if_missing)
    ("fake/archive (4)", True,  0),
]

# ---------- helpers ----------
TITLE_CANDIDATES = ["title","headline","headlines","article_title","story_title","subject"]
TEXT_CANDIDATES  = ["text","content","article","body","full_text","news","message","raw_text","paragraphs","desc","description"]
LABEL_CANDIDATES = ["label","labels","class","target","category","verdict","is_fake","is_real","truth","type","news_type","label_name"]

def normalize_cols(df):
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    for c in df.columns:
        if pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip()
    return df

def pick_first(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def clean_ws(s):
    if pd.isna(s): return ""
    s = str(s)
    s = s.replace("\u201c",'"').replace("\u201d",'"').replace("\u2019","'")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def norm_title(s):
    if pd.isna(s): return ""
    s = str(s).strip()
    # turn placeholders into empty
    return "" if re.fullmatch(r"(nan|none|no\s*title|notitle|n/?a|null|-)", s, flags=re.I) else s

def map_label_series(s):
    s = s.astype(str).str.strip().str.lower()
    mapping = {"real":1,"true":1,"legit":1,"genuine":1,"1":1,"yes":1,"y":1,
               "fake":0,"false":0,"bogus":0,"hoax":0,"0":0,"no":0,"n":0}
    out = s.map(mapping)
    if out.isna().any():
        out_num = pd.to_numeric(s[out.isna()], errors="coerce")
        out.loc[out.isna()] = out_num
    return pd.to_numeric(out, errors="coerce")

def read_table(path):
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".csv":
            return pd.read_csv(path, sep=None, engine="python", on_bad_lines="skip")
        if ext == ".tsv":
            return pd.read_csv(path, sep="\t", engine="python", on_bad_lines="skip")
        if ext in [".xlsx",".xls"]:
            sheets = pd.read_excel(path, sheet_name=None)
            if not sheets: return None
            return max(sheets.values(), key=lambda d: len(d))
        # fallback
        return pd.read_csv(path, engine="python", on_bad_lines="skip")
    except Exception:
        # fallback: headerless single column
        try:
            df = pd.read_csv(path, header=None, engine="python", on_bad_lines="skip")
            df = df.rename(columns={0:"text"})
            return df
        except Exception as e2:
            print(f"[read error] {path}: {e2}")
            return None

def standardize(df_raw, trust_existing_label=False, default_label=None):
    df = normalize_cols(df_raw)

    text_col  = pick_first(df, TEXT_CANDIDATES)
    title_col = pick_first(df, TITLE_CANDIDATES)
    label_col = pick_first(df, LABEL_CANDIDATES) if trust_existing_label else pick_first(df, ["label"])

    # One-column → treat as text
    if len(df.columns) == 1 and text_col is None:
        text_col = df.columns[0]

    # If still no text but title exists, use title as text
    if text_col is None and title_col is not None:
        text_col, title_col = title_col, None

    out = pd.DataFrame()
    out["text"]  = df[text_col] if (text_col and text_col in df.columns) else ""
    out["title"] = df[title_col] if (title_col and title_col in df.columns) else ""

    # Label logic
    if trust_existing_label and label_col and label_col in df.columns:
        out["label"] = map_label_series(df[label_col])
    else:
        if label_col and label_col in df.columns:
            out["label"] = map_label_series(df[label_col])
        else:
            out["label"] = default_label  # may be None for archive (2), but we trust it has label

    # Clean fields
    out["text"]  = out["text"].map(clean_ws)
    out["title"] = out["title"].map(norm_title)

    # Keep rows with label 0/1 and non-empty text (very lenient filter)
    out["label"] = pd.to_numeric(out["label"], errors="coerce")
    out = out[out["label"].isin([0,1])].copy()
    out = out[out["text"].str.len() >= 1].copy()

    return out[["title","text","label"]]

def list_tables(folder_abs):
    out = []
    for fn in os.listdir(folder_abs):
        p = os.path.join(folder_abs, fn)
        if not os.path.isfile(p): continue
        if os.path.splitext(p)[1].lower() in (".csv",".tsv",".xlsx",".xls"):
            out.append(p)
    return sorted(out)

# ---------- process exactly the two folders ----------
for subpath, trust_label, default_lbl in FOLDERS:
    in_dir  = os.path.join(INPUT_ROOT,  subpath)
    out_dir = os.path.join(OUTPUT_ROOT, subpath)
    os.makedirs(out_dir, exist_ok=True)

    if not os.path.isdir(in_dir):
        print(f"[warn] input folder not found: {in_dir}")
        continue

    files = list_tables(in_dir)
    if not files:
        print(f"[warn] no tables in: {in_dir}")
        continue

    print(f"\nProcessing {in_dir} -> {out_dir}")
    for src in files:
        rel_name = os.path.basename(src)
        df_raw = read_table(src)
        if df_raw is None or len(df_raw)==0:
            print("  skip empty:", rel_name); continue

        std = standardize(df_raw, trust_existing_label=trust_label, default_label=default_lbl)
        if std.empty:
            print("  skip no usable rows:", rel_name); continue

        dst = os.path.join(out_dir, os.path.splitext(rel_name)[0] + ".csv")
        std.to_csv(dst, index=False)
        print(f"  ✓ wrote {len(std)} rows -> {dst}")

print("\nDone.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Processing /content/drive/MyDrive/news_raw/fake/archive (2) -> /content/drive/MyDrive/news_clean1/fake/archive (2)
  skip no usable rows: FakeNewsNet.csv

Processing /content/drive/MyDrive/news_raw/fake/archive (4) -> /content/drive/MyDrive/news_clean1/fake/archive (4)
  ✓ wrote 19 rows -> /content/drive/MyDrive/news_clean1/fake/archive (4)/fake.csv

Done.


In [ ]:
# ===========================
# Clean ONLY two folders:
#   1) news_raw/fake/archive (2)  -> trust existing label (1=real,0=fake)
#   2) news_raw/fake/archive (4)  -> keep title,text,label; if label missing -> 0
#      Drop ONLY rows with empty text
# ===========================
import os, re
import pandas as pd

# from google.colab import drive
# drive.mount('/content/drive')

RAW_ROOT    = "/content/drive/MyDrive/news_raw"
CLEAN_ROOT  = "/content/drive/MyDrive/news_clean1"

SRC_ARCH2   = os.path.join(RAW_ROOT,  "fake", "archive (2)")
OUT_ARCH2   = os.path.join(CLEAN_ROOT,"fake", "archive (2)")
SRC_ARCH4   = os.path.join(RAW_ROOT,  "fake", "archive (4)")
OUT_ARCH4   = os.path.join(CLEAN_ROOT,"fake", "archive (4)")

os.makedirs(OUT_ARCH2, exist_ok=True)
os.makedirs(OUT_ARCH4, exist_ok=True)

TITLE_CANDIDATES = ["title","headline","headlines","article_title","story_title","subject"]
TEXT_CANDIDATES  = ["text","content","article","body","full_text","news","message","raw_text","paragraphs","desc","description"]
LABEL_NAMES      = ["label"]  # we only trust 'label' explicitly (1=real,0=fake)

def normalize_cols(df):
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    for c in df.columns:
        if pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip()
    return df

def first_col(df, cands):
    for c in cands:
        if c in df.columns:
            return c
    return None

def clean_ws(s):
    if pd.isna(s): return ""
    s = str(s)
    s = s.replace("\u201c",'"').replace("\u201d",'"').replace("\u2019","'")
    return re.sub(r"\s+", " ", s).strip()

def norm_title(s):
    if pd.isna(s): return ""
    s = str(s).strip()
    return "" if re.fullmatch(r"(nan|none|no\s*title|notitle|n/?a|null|-)", s, flags=re.I) else s

def read_table_any(path):
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".csv":
            return pd.read_csv(path, sep=None, engine="python", on_bad_lines="skip")
        if ext == ".tsv":
            return pd.read_csv(path, sep="\t", engine="python", on_bad_lines="skip")
        if ext in [".xlsx",".xls"]:
            sheets = pd.read_excel(path, sheet_name=None)
            if not sheets: return None
            return max(sheets.values(), key=lambda d: len(d))
        # fallback
        return pd.read_csv(path, engine="python", on_bad_lines="skip")
    except Exception:
        # try headerless one-column
        try:
            df = pd.read_csv(path, header=None, engine="python", on_bad_lines="skip")
            df = df.rename(columns={0:"text"})
            return df
        except Exception as e:
            print(f"[read error] {path}: {e}")
            return None

def map_label_series(s):
    s = s.astype(str).str.strip().str.lower()
    mapping = {"real":1,"true":1,"legit":1,"genuine":1,"1":1,"yes":1,"y":1,
               "fake":0,"false":0,"bogus":0,"hoax":0,"0":0,"no":0,"n":0}
    out = s.map(mapping)
    if out.isna().any():
        out_num = pd.to_numeric(s[out.isna()], errors="coerce")
        out.loc[out.isna()] = out_num
    return pd.to_numeric(out, errors="coerce")

def clean_archive2_file(src_path, dst_path):
    """Trust the existing 'label' column (1=real, 0=fake)."""
    df_raw = read_table_any(src_path)
    if df_raw is None or len(df_raw)==0:
        print("skip empty:", src_path); return
    df = normalize_cols(df_raw)

    text_col  = first_col(df, TEXT_CANDIDATES)
    title_col = first_col(df, TITLE_CANDIDATES)
    label_col = first_col(df, LABEL_NAMES)  # must be 'label'

    # one-column fallback -> treat as text
    if len(df.columns) == 1 and text_col is None:
        text_col = df.columns[0]

    if text_col is None and title_col is not None:
        text_col, title_col = title_col, None

    out = pd.DataFrame()
    out["text"]  = df[text_col] if (text_col and text_col in df.columns) else ""
    out["title"] = df[title_col] if (title_col and title_col in df.columns) else ""

    if label_col and label_col in df.columns:
        out["label"] = map_label_series(df[label_col])
    else:
        print(f"WARNING: {os.path.basename(src_path)} has no 'label' column; skipping to avoid wrong labels.")
        return

    # Normalize fields
    out["text"]  = out["text"].map(clean_ws)
    out["title"] = out["title"].map(norm_title)

    # Keep: label strictly 0/1; text must be non-empty
    out["label"] = pd.to_numeric(out["label"], errors="coerce")
    out = out[out["label"].isin([0,1])].copy()
    out = out[out["text"].str.strip() != ""].copy()

    out = out[["title","text","label"]]
    if len(out)==0:
        print("no usable rows:", src_path); return
    out.to_csv(dst_path, index=False)
    print(f"✓ archive (2): {os.path.basename(src_path)} -> {len(out)} rows -> {dst_path}")

def clean_archive4_file(src_path, dst_path):
    """Extract title,text,label; if label missing set 0. Drop ONLY empty-text rows."""
    df_raw = read_table_any(src_path)
    if df_raw is None or len(df_raw)==0:
        print("skip empty:", src_path); return
    df = normalize_cols(df_raw)

    text_col  = first_col(df, TEXT_CANDIDATES)
    title_col = first_col(df, TITLE_CANDIDATES)
    label_col = first_col(df, LABEL_NAMES)  # may be absent

    if len(df.columns) == 1 and text_col is None:
        text_col = df.columns[0]
    if text_col is None and title_col is not None:
        text_col, title_col = title_col, None

    out = pd.DataFrame()
    out["text"]  = df[text_col] if (text_col and text_col in df.columns) else ""
    out["title"] = df[title_col] if (title_col and title_col in df.columns) else ""

    if label_col and label_col in df.columns:
        out["label"] = map_label_series(df[label_col])
        out["label"] = pd.to_numeric(out["label"], errors="coerce")
        # if label has non 0/1 after mapping, since this is under 'fake', set missing to 0
        out["label"] = out["label"].where(out["label"].isin([0,1]), 0)
    else:
        out["label"] = 0  # under /fake/, default to fake

    # Normalize fields
    out["text"]  = out["text"].map(clean_ws)
    out["title"] = out["title"].map(norm_title)

    # Drop ONLY rows with empty text
    out = out[out["text"].str.strip() != ""].copy()

    out = out[["title","text","label"]]
    if len(out)==0:
        print("no usable rows:", src_path); return
    out.to_csv(dst_path, index=False)
    print(f"✓ archive (4): {os.path.basename(src_path)} -> {len(out)} rows -> {dst_path}")

# -------- run ONLY for these two folders --------
def list_tables(folder):
    if not os.path.isdir(folder): return []
    out = []
    for fn in os.listdir(folder):
        p = os.path.join(folder, fn)
        if os.path.isfile(p) and os.path.splitext(fn)[1].lower() in (".csv",".tsv",".xlsx",".xls"):
            out.append(p)
    return sorted(out)

# archive (2)
for src in list_tables(SRC_ARCH2):
    dst = os.path.join(OUT_ARCH2, os.path.splitext(os.path.basename(src))[0] + ".csv")
    clean_archive2_file(src, dst)

# archive (4)
for src in list_tables(SRC_ARCH4):
    dst = os.path.join(OUT_ARCH4, os.path.splitext(os.path.basename(src))[0] + ".csv")
    clean_archive4_file(src, dst)


✓ archive (4): fake.csv -> 12867 rows -> /content/drive/MyDrive/news_clean1/fake/archive (4)/fake.csv


archive (2) that has real as label column

In [ ]:
# ================================
# Re-clean ONLY: news_raw/fake/archive (2)
# Trust label column named "real" (1=real, 0=fake)
# Output ONLY: title,text,label
# ================================
import os, re
import pandas as pd

# from google.colab import drive
# drive.mount('/content/drive')

RAW_ROOT   = "/content/drive/MyDrive/news_raw"
CLEAN_ROOT = "/content/drive/MyDrive/news_clean1"

SRC_DIR = os.path.join(RAW_ROOT,  "fake", "archive (2)")
OUT_DIR = os.path.join(CLEAN_ROOT,"fake", "archive (2)")
os.makedirs(OUT_DIR, exist_ok=True)

TITLE_CANDIDATES = ["title","headline","headlines","article_title","story_title","subject"]
TEXT_CANDIDATES  = ["text","content","article","body","full_text","news","message","raw_text","paragraphs","desc","description"]

def normalize_cols(df):
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    for c in df.columns:
        if pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip()
    return df

def first_col(df, cands):
    for c in cands:
        if c in df.columns:
            return c
    return None

def clean_ws(s):
    if pd.isna(s): return ""
    s = str(s)
    s = s.replace("\u201c",'"').replace("\u201d",'"').replace("\u2019","'")
    return re.sub(r"\s+", " ", s).strip()

def norm_title(s):
    if pd.isna(s): return ""
    s = str(s).strip()
    return "" if re.fullmatch(r"(nan|none|no\s*title|notitle|n/?a|null|-)", s, flags=re.I) else s

def read_table_any(path):
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".csv":
            return pd.read_csv(path, sep=None, engine="python", on_bad_lines="skip")
        if ext == ".tsv":
            return pd.read_csv(path, sep="\t", engine="python", on_bad_lines="skip")
        if ext in [".xlsx",".xls"]:
            sheets = pd.read_excel(path, sheet_name=None)
            if not sheets: return None
            return max(sheets.values(), key=lambda d: len(d))
        # fallback
        return pd.read_csv(path, engine="python", on_bad_lines="skip")
    except Exception:
        # headerless single-column fallback
        try:
            df = pd.read_csv(path, header=None, engine="python", on_bad_lines="skip")
            df = df.rename(columns={0:"text"})
            return df
        except Exception as e:
            print(f"[read error] {path}: {e}")
            return None

def map_real_to_label(s):
    """
    Convert the 'real' column to numeric label: 1=real, 0=fake.
    Accepts strings/numbers, coerces to 0/1, drops others.
    """
    s = s.astype(str).str.strip().str.lower()
    mapping = {"1":1, "0":0, "real":1, "true":1, "fake":0, "false":0, "yes":1, "no":0, "y":1, "n":0}
    out = s.map(mapping)
    # numeric fallback
    if out.isna().any():
        out_num = pd.to_numeric(s[out.isna()], errors="coerce")
        out.loc[out.isna()] = out_num
    out = pd.to_numeric(out, errors="coerce").astype("Int64")
    return out

def list_tables(folder):
    if not os.path.isdir(folder): return []
    out = []
    for fn in os.listdir(folder):
        p = os.path.join(folder, fn)
        if os.path.isfile(p) and os.path.splitext(fn)[1].lower() in (".csv",".tsv",".xlsx",".xls"):
            out.append(p)
    return sorted(out)

def clean_file(src_path, dst_path):
    df_raw = read_table_any(src_path)
    if df_raw is None or len(df_raw)==0:
        print("skip empty:", os.path.basename(src_path)); return
    df = normalize_cols(df_raw)

    # find columns
    text_col  = first_col(df, TEXT_CANDIDATES)
    title_col = first_col(df, TITLE_CANDIDATES)

    # MUST have 'real' column for labels here
    if "real" not in df.columns:
        print(f"WARNING: '{os.path.basename(src_path)}' has no 'real' column. Skipped to avoid wrong labels.")
        return

    # one-column fallback -> treat as text
    if len(df.columns) == 1 and text_col is None:
        text_col = df.columns[0]

    # if still no text but title exists, use title as text
    if text_col is None and title_col is not None:
        text_col, title_col = title_col, None

    out = pd.DataFrame()
    out["text"]  = df[text_col] if (text_col and text_col in df.columns) else ""
    out["title"] = df[title_col] if (title_col and title_col in df.columns) else ""

    # map 'real' -> label
    out["label"] = map_real_to_label(df["real"])

    # clean fields
    out["text"]  = out["text"].map(clean_ws)
    out["title"] = out["title"].map(norm_title)

    # keep only rows with non-empty text and label in {0,1}
    out = out[out["text"].str.strip() != ""].copy()
    out = out[out["label"].isin([0,1])].copy()

    # final order
    out = out[["title","text","label"]]
    if len(out)==0:
        print("no usable rows:", os.path.basename(src_path)); return

    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    out.to_csv(dst_path, index=False)
    print(f"✓ {os.path.basename(src_path)} -> {len(out)} rows -> {dst_path}")

# Run for all tables in archive (2)
files = list_tables(SRC_DIR)
print(f"Found {len(files)} file(s) in {SRC_DIR}")
for src in files:
    dst = os.path.join(OUT_DIR, os.path.splitext(os.path.basename(src))[0] + ".csv")
    clean_file(src, dst)


Found 1 file(s) in /content/drive/MyDrive/news_raw/fake/archive (2)
✓ FakeNewsNet.csv -> 23196 rows -> /content/drive/MyDrive/news_clean1/fake/archive (2)/FakeNewsNet.csv


real data preprocess

In [ ]:
# Clean ONLY: news_raw/real/archive/True.csv  ->  label all rows as 1 (real)
import os, re
import pandas as pd

# from google.colab import drive
# drive.mount('/content/drive')

RAW_ROOT   = "/content/drive/MyDrive/news_raw"
CLEAN_ROOT = "/content/drive/MyDrive/news_clean1"

SRC_DIR = os.path.join(RAW_ROOT,  "real", "archive")
OUT_DIR = os.path.join(CLEAN_ROOT,"real", "archive")
os.makedirs(OUT_DIR, exist_ok=True)

SRC_FILE = os.path.join(SRC_DIR, "True.csv")   # change if your filename differs in case

TITLE_CANDIDATES = ["title","headline","headlines","article_title","story_title","subject"]
TEXT_CANDIDATES  = ["text","content","article","body","full_text","news","message","raw_text","paragraphs","desc","description"]

def normalize_cols(df):
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    for c in df.columns:
        if pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip()
    return df

def first_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def clean_ws(s):
    if pd.isna(s): return ""
    s = str(s)
    s = s.replace("\u201c",'"').replace("\u201d",'"').replace("\u2019","'")
    return re.sub(r"\s+", " ", s).strip()

def norm_title(s):
    if pd.isna(s): return ""
    s = str(s).strip()
    # turn placeholders into empty string
    return "" if re.fullmatch(r"(nan|none|no\s*title|notitle|n/?a|null|-)", s, flags=re.I) else s

def read_table_any(path):
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".csv":
            return pd.read_csv(path, sep=None, engine="python", on_bad_lines="skip")
        if ext == ".tsv":
            return pd.read_csv(path, sep="\t", engine="python", on_bad_lines="skip")
        if ext in [".xlsx",".xls"]:
            sheets = pd.read_excel(path, sheet_name=None)
            if not sheets: return None
            return max(sheets.values(), key=lambda d: len(d))
        # fallback
        return pd.read_csv(path, engine="python", on_bad_lines="skip")
    except Exception:
        # headerless single-column fallback
        try:
            df = pd.read_csv(path, header=None, engine="python", on_bad_lines="skip")
            df = df.rename(columns={0:"text"})
            return df
        except Exception as e:
            print(f"[read error] {path}: {e}")
            return None

def clean_true_csv(src_path, dst_path):
    df_raw = read_table_any(src_path)
    if df_raw is None or len(df_raw)==0:
        print("Skip empty/unreadable:", os.path.basename(src_path)); return
    df = normalize_cols(df_raw)

    text_col  = first_col(df, TEXT_CANDIDATES)
    title_col = first_col(df, TITLE_CANDIDATES)

    # one-column fallback → treat as text
    if len(df.columns) == 1 and text_col is None:
        text_col = df.columns[0]

    # if still no text but title exists, use title as text
    if text_col is None and title_col is not None:
        text_col, title_col = title_col, None

    out = pd.DataFrame()
    out["text"]  = df[text_col] if (text_col and text_col in df.columns) else ""
    out["title"] = df[title_col] if (title_col and title_col in df.columns) else ""

    # Clean fields
    out["text"]  = out["text"].map(clean_ws)
    out["title"] = out["title"].map(norm_title)

    # Drop rows with empty text
    out = out[out["text"].str.strip() != ""].copy()

    # Label all as real = 1
    out["label"] = 1

    # Final order & save
    out = out[["title","text","label"]]
    if len(out)==0:
        print("No usable rows after cleaning.")
        return

    out.to_csv(dst_path, index=False)
    print(f"✓ {os.path.basename(src_path)} -> {len(out)} rows -> {dst_path}")

# Run
DST_FILE = os.path.join(OUT_DIR, "True.csv")  # same name in cleaned folder
clean_true_csv(SRC_FILE, DST_FILE)


✓ True.csv -> 21416 rows -> /content/drive/MyDrive/news_clean1/real/archive/True.csv
